# 🖼️ Alt Text Generator — GPT-4o Vision
**SEO Tooling**

Generates SEO-optimised alt attributes at scale using GPT-4o Vision + live page context crawling + an optional keyword list.

---
### Workflow
1. **Cell 1** — Install dependencies
2. **Cell 2** — 🔐 Add your OpenAI API key (paste into a hidden field *or* upload a .txt file)
3. **Cell 3** — Configuration & shared functions
4. **Cell 4** — Upload image/page CSV
5. **Cell 5** — 🔑 Keywords (optional but recommended)
6. **Cell 6** — 🧪 TEST RUN (5 rows) — review before committing
7. **Cell 7** — ✅ FULL RUN (all rows)

### Required CSV columns
| Column | Description |
|---|---|
| `image_url` | Full URL to the image file |
| `page_url` | Full URL of the page hosting the image |

### About the API key
The key is **never stored in this notebook**. It is held in memory for the current Colab session only, so you can safely share or save the notebook. When the runtime disconnects or resets, you'll re-run Cell 2.

In [ ]:
# ============================================================
#  CELL 1 — Install Dependencies
#  Run once per Colab session
# ============================================================
!pip install -q openai requests beautifulsoup4 pandas
print("✅ Dependencies installed")

In [ ]:
#@title 🔐 CELL 2 — Add OpenAI API Key { display-mode: "form" }
#@markdown Choose how to provide your key. It is kept in memory for this session only and is never written into the notebook.
KEY_METHOD = "Paste (hidden text field)"  #@param ["Paste (hidden text field)", "Upload .txt file"]

import os
import re
from getpass import getpass
from google.colab import files
from openai import OpenAI, AuthenticationError

KEY_PATTERN = re.compile(r"sk-[A-Za-z0-9_\-]{20,}")

def _extract_key(text: str) -> str:
    """Pulls an OpenAI key out of raw text (handles quotes, whitespace, 'OPENAI_API_KEY=...' lines)."""
    match = KEY_PATTERN.search(text or "")
    return match.group(0) if match else ""

_raw = ""
if KEY_METHOD.startswith("Paste"):
    # getpass renders a masked input box and does not save the value in cell output
    _raw = getpass("Paste your OpenAI API key and press Enter: ")
else:
    print("📂 Upload a .txt file containing your API key (the file can contain just the key or OPENAI_API_KEY=sk-...)")
    _uploaded = files.upload()
    if _uploaded:
        _fname = list(_uploaded.keys())[0]
        _raw = _uploaded[_fname].decode("utf-8", errors="ignore")
        try:
            os.remove(_fname)  # remove the key file from the Colab disk right away
        except OSError:
            pass

_key = _extract_key(_raw)
_raw = ""  # drop the raw input

if not _key:
    raise ValueError("❌ No valid OpenAI key found (keys start with 'sk-'). Re-run this cell and try again.")

# Verify the key with a lightweight, free call before anything else runs
_test_client = OpenAI(api_key=_key)
try:
    _test_client.models.list()
except AuthenticationError:
    _key = ""
    raise ValueError("❌ OpenAI rejected this key (authentication failed). Check it and re-run this cell.")

os.environ["OPENAI_API_KEY"] = _key
client = _test_client
print(f"✅ API key verified and loaded for this session (ending in …{_key[-4:]})")
print("   The key lives in memory only — it is not saved in this notebook.")
_key = ""  # clear the local copy

In [ ]:
# ============================================================
#  CELL 3 — Configuration & Shared Functions
#  Run after Cell 2 (needs the API key loaded)
# ============================================================
import pandas as pd
import requests
import base64
import time
import re
from bs4 import BeautifulSoup
from google.colab import files
from IPython.display import display

# ── USER SETTINGS ────────────────────────────────────────────
ALT_TEXT_MAX_CHARS  = 125     # Hard ceiling; industry best practice
BATCH_DELAY_SECONDS = 0.5     # Pause between API calls (rate limit safety)
REQUEST_TIMEOUT     = 10      # Seconds for page/image HTTP requests
IMAGE_DETAIL        = "low"   # "low" = ~$0.002/image | "high" = better for complex product shots
TEST_ROWS           = 5       # Number of rows used in the test run
# ─────────────────────────────────────────────────────────────

CRAWL_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AltTextBot/1.0; +https://github.com/jasonmcgovern/gallery)"
}

# Keyword list — populated by Cell 5. Do not edit here.
SITE_KEYWORDS = []


# ── PAGE CRAWLER ─────────────────────────────────────────────
def crawl_page_context(page_url: str) -> dict:
    """Fetches page and extracts title, meta description, H1."""
    result = {"page_title": "", "meta_description": "", "h1": "", "crawl_status": ""}
    try:
        resp = requests.get(page_url, headers=CRAWL_HEADERS,
                            timeout=REQUEST_TIMEOUT, allow_redirects=True)
        result["crawl_status"] = str(resp.status_code)
        if resp.status_code != 200:
            return result
        soup = BeautifulSoup(resp.text, "html.parser")
        title_tag = soup.find("title")
        result["page_title"] = title_tag.get_text(strip=True) if title_tag else ""
        meta = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
        if meta:
            result["meta_description"] = meta.get("content", "").strip()
        h1_tag = soup.find("h1")
        result["h1"] = h1_tag.get_text(strip=True) if h1_tag else ""
    except requests.exceptions.RequestException as e:
        result["crawl_status"] = f"Error: {str(e)[:80]}"
    return result


# ── IMAGE FETCHER ─────────────────────────────────────────────
def fetch_image_as_base64(image_url: str):
    """Downloads image and returns (base64_string, media_type) or (None, error)."""
    try:
        resp = requests.get(image_url, headers=CRAWL_HEADERS, timeout=REQUEST_TIMEOUT)
        if resp.status_code != 200:
            return None, f"HTTP {resp.status_code}"
        content_type = resp.headers.get("Content-Type", "image/jpeg").split(";")[0].strip()
        type_map = {"image/jpg": "image/jpeg", "image/pjpeg": "image/jpeg", "image/x-png": "image/png"}
        media_type = type_map.get(content_type, content_type)
        if media_type not in ("image/jpeg", "image/png", "image/gif", "image/webp"):
            return None, f"Unsupported type: {media_type}"
        b64 = base64.standard_b64encode(resp.content).decode("utf-8")
        return b64, media_type
    except requests.exceptions.RequestException as e:
        return None, str(e)[:80]


# ── ALT TEXT GENERATOR ───────────────────────────────────────
def generate_alt_text(image_url, page_title, meta_description, h1, keywords=None):
    """
    Sends image + page context + optional keyword list to GPT-4o Vision.
    Returns (alt_text, keyword_used, flag).
    """
    context_parts = []
    if page_title:       context_parts.append(f"Page Title: {page_title}")
    if h1:               context_parts.append(f"H1: {h1}")
    if meta_description: context_parts.append(f"Meta Description: {meta_description[:200]}")
    context_str = "\n".join(context_parts) if context_parts else "No page context available."

    # ── Build keyword section of the prompt ──────────────────
    if keywords:
        kw_sample = keywords[:30]  # cap to avoid token bloat
        kw_block = (
            "KEYWORD REFERENCE LIST:\n"
            "The following are target SEO keywords for this site. "
            "If ONE keyword from this list naturally and accurately describes "
            "something visible in the image, incorporate it. "
            "If no keyword fits organically, ignore the list entirely — "
            "do NOT force a keyword into alt text where it does not belong.\n"
            + "\n".join(f"- {kw}" for kw in kw_sample)
        )
    else:
        kw_block = ""

    system_prompt = """You are an SEO and accessibility specialist writing alt attributes for website images.

RULES:
1. Describe what is literally visible in the image — be specific and concrete.
2. Incorporate the page context naturally only if it genuinely applies to the image content.
3. Do NOT keyword-stuff. At most one keyword reference per alt text.
4. Do NOT start with "Image of", "Photo of", or "Picture of".
5. Keep output to 125 characters or fewer (hard limit).
6. For decorative/purely background images, return exactly: DECORATIVE
7. After the alt text, on a new line, write: KEYWORD_USED: <the keyword you used, or NONE>

Output format (two lines only):
<alt text here>
KEYWORD_USED: <keyword or NONE>"""

    user_prompt = f"""Page context:
{context_str}

{kw_block}

Write the alt attribute for the image provided.""".strip()

    b64_data, media_type_or_error = fetch_image_as_base64(image_url)
    if b64_data is None:
        return "", "NONE", f"Image fetch failed: {media_type_or_error}"

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {
                        "url": f"data:{media_type_or_error};base64,{b64_data}",
                        "detail": IMAGE_DETAIL
                    }},
                    {"type": "text", "text": user_prompt}
                ]}
            ],
            max_tokens=120,
            temperature=0
        )

        raw = response.choices[0].message.content.strip()

        # ── Parse the two-line response ───────────────────────
        keyword_used = "NONE"
        if "KEYWORD_USED:" in raw:
            lines = raw.split("\n")
            kw_line = next((l for l in lines if l.startswith("KEYWORD_USED:")), "")
            keyword_used = kw_line.replace("KEYWORD_USED:", "").strip()
            alt_text = "\n".join(l for l in lines if not l.startswith("KEYWORD_USED:")).strip()
        else:
            alt_text = raw

        alt_text = alt_text.strip('"\'')

        # ── Flag logic ────────────────────────────────────────
        flag = ""
        if alt_text == "DECORATIVE":
            flag = "Decorative — consider empty alt or role=presentation"
        elif len(alt_text) > ALT_TEXT_MAX_CHARS:
            flag = f"Over {ALT_TEXT_MAX_CHARS} chars — needs trimming"

        return alt_text, keyword_used, flag

    except Exception as e:
        return "", "NONE", f"API error: {str(e)[:80]}"


# ── CORE PIPELINE ─────────────────────────────────────────────
def run_pipeline(df, label=""):
    """Processes a dataframe of image/page URLs. Returns enriched dataframe."""
    if "client" not in globals():
        raise RuntimeError("No API key loaded — run Cell 2 first.")

    df = df.copy()
    df.columns = df.columns.str.lower()

    required_cols = {"image_url", "page_url"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing required columns: {missing}. Found: {list(df.columns)}")

    for col in ["page_title", "meta_description", "h1", "crawl_status",
                "alt_text", "keyword_used", "flag"]:
        df[col] = ""
    df["char_count"] = 0

    page_cache = {}
    total = len(df)
    tag = f"[{label}] " if label else ""
    kw_active = len(SITE_KEYWORDS) > 0

    for idx, row in df.iterrows():
        image_url = str(row["image_url"]).strip()
        page_url  = str(row["page_url"]).strip()
        print(f"{tag}[{df.index.get_loc(idx)+1}/{total}] {image_url[:90]}")

        if page_url not in page_cache:
            print(f"  ↳ Crawling: {page_url[:90]}")
            page_cache[page_url] = crawl_page_context(page_url)

        ctx = page_cache[page_url]
        df.at[idx, "page_title"]       = ctx["page_title"]
        df.at[idx, "meta_description"] = ctx["meta_description"]
        df.at[idx, "h1"]               = ctx["h1"]
        df.at[idx, "crawl_status"]     = ctx["crawl_status"]

        alt_text, keyword_used, flag = generate_alt_text(
            image_url,
            ctx["page_title"],
            ctx["meta_description"],
            ctx["h1"],
            keywords=SITE_KEYWORDS if kw_active else None
        )

        df.at[idx, "alt_text"]     = alt_text
        df.at[idx, "keyword_used"] = keyword_used
        df.at[idx, "char_count"]   = len(alt_text)
        df.at[idx, "flag"]         = flag

        kw_note = f" [kw: {keyword_used}]" if keyword_used and keyword_used != "NONE" else ""
        status_icon = "⚠️ " if flag else "✓"
        print(f"  {status_icon} {alt_text[:85]}{kw_note}")
        if flag:
            print(f"     ↳ {flag}")

        time.sleep(BATCH_DELAY_SECONDS)

    return df


def print_summary(df, label=""):
    """Prints a summary of results including keyword usage rate."""
    total        = len(df)
    generated    = df[df["alt_text"] != ""].shape[0]
    decorative   = (df["alt_text"] == "DECORATIVE").sum()
    over_limit   = df["flag"].str.contains("chars", na=False).sum()
    failed       = df["flag"].str.contains("failed|error", case=False, na=False).sum()
    avg_chars    = df[df["alt_text"] != ""]["char_count"].mean()
    kw_used      = (df["keyword_used"].notna() & (df["keyword_used"] != "NONE") & (df["keyword_used"] != "")).sum()
    kw_rate      = f"{kw_used}/{generated} ({kw_used/generated*100:.0f}%)" if generated > 0 else "N/A"
    tag = f" — {label}" if label else ""
    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SUMMARY{tag}
  Total rows processed   : {total}
  Alt texts generated    : {generated}
  Keyword incorporated   : {kw_rate}
  Decorative flagged     : {decorative}
  Over {ALT_TEXT_MAX_CHARS}-char limit      : {over_limit}
  Failed (fetch/API)     : {failed}
  Avg character count    : {avg_chars:.0f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")


print("✅ Cell 3 loaded — configuration and functions ready.")
print(f"   Model detail level : {IMAGE_DETAIL}")
print(f"   Max alt text chars : {ALT_TEXT_MAX_CHARS}")
print(f"   Test row count     : {TEST_ROWS}")

In [ ]:
# ============================================================
#  CELL 4 — Upload Image/Page CSV
#  Required before Cell 6 or 7.
# ============================================================
print("📂 Upload your CSV (required columns: image_url, page_url)")
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]

df_full = pd.read_csv(uploaded_filename)
df_full.columns = df_full.columns.str.lower()

print(f"\n✅ Loaded: {uploaded_filename}")
print(f"   Rows        : {len(df_full)}")
print(f"   Columns     : {list(df_full.columns)}")
print(f"   Unique pages: {df_full['page_url'].nunique() if 'page_url' in df_full.columns else 'N/A'}")
print("\nFirst 3 rows preview:")
display(df_full[["image_url", "page_url"]].head(3))

In [ ]:
# ============================================================
#  CELL 5 — 🔑 Keyword List (Optional but Recommended)
#
#  Paste your site's target keywords below, one per line.
#  These are treated as SUGGESTIONS — the model will only
#  use a keyword if it naturally fits what's visible in the
#  image. No forced inclusion.
#
#  Tips:
#  • Include a mix of head terms and long-tail phrases
#  • 20–50 keywords is a good range; more than 30 will be
#    sampled down to avoid token bloat
#  • Order roughly by priority — the list is passed top-down
#  • To run WITHOUT keywords, clear the list below and run
#    this cell anyway (it will confirm 0 keywords loaded)
# ============================================================

SITE_KEYWORDS = """
replace this line with your first keyword
second keyword here
third keyword here
""".strip().splitlines()

# Clean: remove blanks, strip whitespace, deduplicate
SITE_KEYWORDS = list(dict.fromkeys(
    kw.strip() for kw in SITE_KEYWORDS if kw.strip()
))

if SITE_KEYWORDS:
    print(f"✅ {len(SITE_KEYWORDS)} keywords loaded")
    if len(SITE_KEYWORDS) > 30:
        print("   ⚠️  More than 30 keywords — only the first 30 will be passed to the prompt per image")
    print("\nKeyword list:")
    for i, kw in enumerate(SITE_KEYWORDS, 1):
        print(f"   {i:>2}. {kw}")
else:
    print("ℹ️  No keywords loaded — running without keyword guidance.")
    print("   Alt texts will rely on page context (title, H1, meta) only.")

In [ ]:
# ============================================================
#  CELL 6 — 🧪 TEST RUN (first N rows)
#
#  Run this BEFORE the full batch to:
#    • Confirm image URLs are reachable
#    • Verify page crawls return title/H1/meta
#    • Check keyword incorporation feels natural
#    • Catch any API key or format issues early
#
#  Adjust TEST_ROWS in Cell 3 if you want more/fewer samples.
#  Cost: ~$0.01–$0.02 for 5 rows at detail=low
# ============================================================
print(f"🧪 TEST RUN — processing first {TEST_ROWS} rows")
print(f"   Keywords active: {'Yes (' + str(len(SITE_KEYWORDS)) + ' keywords)' if SITE_KEYWORDS else 'No'}")
print("─" * 60 + "\n")

df_test = run_pipeline(df_full.head(TEST_ROWS).reset_index(drop=True), label="TEST")
print_summary(df_test, label="TEST RUN")

print("\n📋 Full test results:")

def highlight_row(row):
    """Row-level styling for the test output table."""
    styles = [""] * len(row)
    flag = str(row.get("flag", ""))
    kw   = str(row.get("keyword_used", ""))
    if "failed" in flag.lower() or "error" in flag.lower():
        styles = ["background-color: #f8d7da"] * len(row)
    elif "Over" in flag:
        styles = ["background-color: #fff3cd"] * len(row)
    elif kw and kw not in ("NONE", ""):
        styles = ["background-color: #d4edda"] * len(row)  # green = keyword used
    return styles

display(
    df_test[[
        "image_url", "page_title", "h1",
        "alt_text", "keyword_used", "char_count", "flag"
    ]].style
    .apply(highlight_row, axis=1)
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "max-width": "300px"})
)

print("\n" + "─" * 60)
print("👆 Review the results above before running Cell 7 (full batch).")
print("   Things to check:")
print("   🟢 Green rows  = keyword incorporated naturally")
print("   🟡 Yellow rows = alt text over 125 chars (needs trimming)")
print("   🔴 Red rows    = image fetch or API failure")
print("   • Does the keyword feel natural or forced?")
print("   • Are page_title and h1 being pulled correctly?")
print("   • Any unexpected DECORATIVE tags?")
print("   If keywords feel forced, try removing lower-quality terms from Cell 5.")

In [ ]:
# ============================================================
#  CELL 7 — ✅ FULL RUN
#
#  Only run this after reviewing Cell 6 output.
#  Processes all rows in your CSV.
#  Estimated time  : ~10–14 min for 1,200 images
#  Estimated cost  : ~$2.50–$3.50 at detail=low
# ============================================================
print(f"✅ FULL RUN — processing all {len(df_full)} rows")
print(f"   Keywords active    : {'Yes (' + str(len(SITE_KEYWORDS)) + ' keywords)' if SITE_KEYWORDS else 'No'}")
print(f"   Estimated time     : ~{round(len(df_full) * 0.7 / 60, 0):.0f}–{round(len(df_full) * 1.0 / 60, 0):.0f} minutes")
print(f"   Estimated cost     : ~${len(df_full) * 0.003:.2f} (gpt-4o, detail={IMAGE_DETAIL})")
print("─" * 60 + "\n")

df_output = run_pipeline(df_full.reset_index(drop=True), label="FULL")
print_summary(df_output, label="FULL RUN")

# ── Keyword usage breakdown ──────────────────────────────────
kw_counts = (
    df_output[df_output["keyword_used"].notna() & (df_output["keyword_used"] != "NONE") & (df_output["keyword_used"] != "")]
    ["keyword_used"]
    .value_counts()
    .reset_index()
)
kw_counts.columns = ["keyword", "times_used"]

if not kw_counts.empty:
    print("\n🔑 Keyword usage breakdown:")
    display(kw_counts)

# ── Save & download ──────────────────────────────────────────
output_filename = "alt_text_output.csv"
df_output.to_csv(output_filename, index=False)
print(f"\n💾 Saved: {output_filename}")

# Preview flagged rows for QA
flagged = df_output[df_output["flag"] != ""]
if len(flagged) > 0:
    print(f"\n⚠️  {len(flagged)} rows need review:")
    display(flagged[["image_url", "alt_text", "char_count", "flag"]].reset_index(drop=True))
else:
    print("\n✅ No flagged rows — all alt texts look clean!")

files.download(output_filename)
print("\n📥 Download started.")